# Southern Ocean zonal means

Volume-weighted zonal-mean sections of potential temperature (θ), salinity (S) and σ2 south of 30°S, averaged
over the `Avg` period in `diag_config.yml`, for the circumpolar Southern Ocean and the five Antarctic sea-ice
sectors (Weddell, Indian, WestPacific, Ross, AmundsenBellingshausen). Computed by `mom6-tools_zonal_means`
(`mom6_tools/compute_zonal_means.py`); black contours are σ2 (kg m⁻³).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%%capture
# comment above line to see details about the run(s) displayed
from misc import *
import numpy as np
import xarray as xr
from matplotlib.ticker import MaxNLocator
from IPython.display import Image, display

In [ ]:
%matplotlib inline

## Model

In [ ]:
# figures written by mom6-tools_zonal_means: one per variable and region
variables = ['thetao', 'so', 'sigma2']
regions = ['Circumpolar', 'Weddell', 'Indian', 'WestPacific', 'Ross', 'AmundsenBellingshausen']
for i in range(len(casename)):
  for var in variables:
    for reg in regions:
      png_file = os.path.join(png_dir, 'zonal_means', '{}_zonal_means_{}_{}.png'.format(casename[i], var, reg))
      if os.path.exists(png_file):
        display(Image(filename=png_file))
      else:
        print('Not found:', png_file)

## Comparison

Each case minus the first case (`Case1`) in `diag_config.yml`. Both cases must be on the same grid (same `z_l`
and `lat` axes), i.e. same model grid and same `-lat` limit when `mom6-tools_zonal_means` was run.

In [ ]:
SIG2_LEVELS = [35.5, 36.0, 36.5, 36.8, 36.9, 37.0, 37.05, 37.1, 37.15]   # sigma2 contours
DTITLE = {'thetao': 'Δθ (°C)', 'so': 'ΔS (psu)', 'sigma2': 'Δσ2 (kg m⁻³)'}

def edges(c):
  """Cell edges from centers (midpoints, with the end cells mirrored)."""
  m = 0.5 * (c[1:] + c[:-1])
  return np.concatenate([[2 * c[0] - m[0]], m, [2 * c[-1] - m[-1]]])

def period(ds):
  """Averaging period of a dataset written by mom6-tools_zonal_means."""
  return 'years {}-{}'.format(int(ds.attrs['year_start']), int(ds.attrs['year_end']))

def check_compatible(ds, ds_ref):
  """Raise if ds and ds_ref are not on the same region/z_l/lat axes."""
  for c in ['region', 'z_l', 'lat']:
    a, b = ds[c].values, ds_ref[c].values
    same = (a.shape == b.shape) and (np.array_equal(a, b) if a.dtype.kind in 'OUS' else np.allclose(a, b))
    if not same:
      raise ValueError('{} differs between the two datasets ({} vs {} points): '
                       'cases must be on the same grid and use the same -lat'.format(c, a.size, b.size))

def plot_diff(ds, ds_ref, var, label, ref_label, lim=None, ncols=3):
  """
  Zonal-mean difference ds minus ds_ref of var, one panel per region.

  Parameters
  ----------
  ds, ds_ref : xarray.Dataset
    Datasets written by mom6-tools_zonal_means (<casename>_zonal_means.nc), on the same grid.
  var : str
    'thetao', 'so' or 'sigma2'.
  label, ref_label : str
    Names of the two cases, used in the title.
  lim : float, optional
    Symmetric color limit. Default is the 95th percentile of |difference| over all regions.
  ncols : int, optional
    Number of panel columns.

  Returns
  -------
  fig : matplotlib Figure. Contours are sigma2 of ds.
  """
  check_compatible(ds, ds_ref)
  diff = ds[var] - ds_ref[var]
  if lim is None:
    lim = np.nanpercentile(np.abs(diff.values), 95)
    lim = lim if lim > 0 else 1.
  regs = ds.region.values
  nrows = int(np.ceil(len(regs) / ncols))
  late = edges(ds.lat.values)
  fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 3.2 * nrows + 1), sharex=True, sharey=True,
                          squeeze=False, layout='constrained')
  for ax, reg in zip(axs.flat, regs):
    pc = ax.pcolormesh(late, ds.z_i.values, diff.sel(region=reg), shading='flat',
                       cmap='RdBu_r', vmin=-lim, vmax=lim)
    cs = ax.contour(ds.lat, ds.z_l, ds.sigma2.sel(region=reg), levels=SIG2_LEVELS, colors='k', linewidths=0.6)
    ax.clabel(cs, fontsize=6, fmt='%g')
    ax.set_title(str(reg), fontsize=11)
    ax.set_ylim(5500, 0)
    ax.set_xlim(late[0], late[-1])
  for ax in axs.flat[len(regs):]:
    ax.set_visible(False)
  for ax in axs[:, 0]:
    ax.set_ylabel('depth (m)')
  for ax in axs[-1, :]:
    ax.set_xlabel('latitude')
  fig.colorbar(pc, ax=axs, orientation='horizontal', shrink=0.5, extend='both', label=DTITLE[var],
               ticks=MaxNLocator(5, symmetric=True))
  fig.suptitle('{} ({}) minus {} ({}); contours: σ2 of {}'.format(label, period(ds), ref_label,
               period(ds_ref), label))
  return fig

In [ ]:
# load the zonal means of each case
ds_zm = []
for path, case in zip(ocn_path, casename):
  fname = os.path.join(os.path.expandvars(path), case + '_zonal_means.nc')
  ds_zm.append(xr.open_dataset(fname))

In [ ]:
if len(ds_zm) < 2:
  print('Only one case in diag_config.yml: nothing to compare.')
for i in range(1, len(ds_zm)):
  for var in ['thetao', 'so', 'sigma2']:
    fig = plot_diff(ds_zm[i], ds_zm[0], var, label[i], label[0])
    plt.show()